In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import mlflow
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline

In [3]:
load_dotenv()
df_path = os.getenv("DATASET_PATH")
file_path = os.path.join(df_path, "data.csv")
df = pd.read_csv(file_path)
pd.set_option('display.max_columns', None)

Task was destroyed but it is pending!
task: <Task pending name='Task-32' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-33' coro=<Kernel.shell_main() running at C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
<frozen _collections_abc>:439: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
Task was destroyed but it is pending!
task: <Task pending name='Task-33' coro=<Kernel.shell_main() running at C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


In [4]:
df.isna().sum()[df.isna().sum()>0]

Unnamed: 32    569
dtype: int64

In [5]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [6]:
df = df.drop("Unnamed: 32", axis=1)

In [7]:
df['diagnosis'].value_counts(normalize=True)

diagnosis
B    0.627417
M    0.372583
Name: proportion, dtype: float64

In [8]:
X = df.drop(['id', 'diagnosis'], axis=1)
y = df['diagnosis']
y = y.map({"B":0, "M":1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "Sklearn_HistGB": HistGradientBoostingClassifier(random_state=42),
    "RandomFores": RandomForestClassifier(random_state=42)}

results = []

In [ ]:
mlflow.set_experiment("5 Models Comparison")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
results.clear()

BASE_OUTPUT_DIR = "model_evaluations"
TEXT_DIR = os.path.join(BASE_OUTPUT_DIR, "text")
IMAGE_DIR = os.path.join(BASE_OUTPUT_DIR, "images")

os.makedirs(TEXT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

for model_name, model in models.items():
    with mlflow.start_run(run_name=model_name):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        probs = pipe.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, probs)
        f1 = f1_score(y_test, preds)
        
        report = classification_report(y_test, preds)
        report_path = os.path.join(TEXT_DIR, f"{model_name}_classification_report.txt")
        with open(report_path, "w") as f:
            f.write(report)

        cm = confusion_matrix(y_test, preds)
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix - {model_name}")
        cm_path = os.path.join(IMAGE_DIR, f"{model_name}_confusion_matrix.png")
        plt.savefig(cm_path)
        plt.close()

        results.append({"model":model_name, "accuracy":acc, "roc_auc_score":roc_auc, "f1_score":f1})
        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc_score", roc_auc)
        mlflow.log_artifact(cm_path, artifact_path="images")
        mlflow.log_artifact(report_path, artifact_path="text")

        mlflow.sklearn.log_model(pipe, artifact_path="model_pipeline", serialization_format="pickle")

KeyboardInterrupt: 

In [18]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", XGBClassifier(random_state=42))])

param_distributions = {
'model__max_depth': np.linspace(3, 10),
'model__learning_rate': np.arange(0.01, 0.1, 0.1),
'model__n_estimators': np.linspace(100, 1000),
'model__subsample': np.arange(0.5, 0.5, 0.25),
'model__colsample_bytree': np.arange(0.5, 0.5, 0.25)
}

grids =  {"F1": GridSearchCV(
    estimator = pipe,
    param_grid=param_grid,
    scoring="f1",
    cv = 5,
    n_jobs=-1,
    return_train_score=True)}
    

In [21]:
mlflow.set_experiment("5 Models Comparison")
mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.sklearn.autolog()
for metric, grid in grids.items():
    with mlflow.start_run(run_name=f"XGBoost_GridSearch_Registry_{metric}"):
        grid.fit(X_train, y_train)
    
        best_model = grid.best_estimator_
    
        preds = best_model.predict(X_test)
        probs = best_model.predict_proba(X_test)[:, 1]
    
        accuracy = accuracy_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, probs)
        f1 = f1_score(y_test, preds)
    
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc_score", roc_auc)
        mlflow.log_metric("f1_score", f1)

        mlflow.sklearn.log_model(best_model, artifact_path="model_pipeline", serialization_format="pickle", registered_model_name="BreastCancerClassifier")
    

2026/07/30 10:53:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
C:\Users\bben2\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(
2026/07/30 10:53:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https:

🏃 View run handsome-ant-781 at: http://127.0.0.1:5000/#/experiments/4/runs/bb8d567e19d5483993d7307d16559931
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run respected-hog-508 at: http://127.0.0.1:5000/#/experiments/4/runs/1ae03b06ae5d435eb7f85272e85fcb13
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run puzzled-horse-307 at: http://127.0.0.1:5000/#/experiments/4/runs/c28dbd1e7d404f858b0fda453cf54952
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/07/30 10:53:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mysterious-vole-7 at: http://127.0.0.1:5000/#/experiments/4/runs/c83272a6e8b14043a170fba8398623d3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run magnificent-kit-771 at: http://127.0.0.1:5000/#/experiments/4/runs/616ad3e9aa80483db2050c17ed9d16cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/07/30 10:53:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'BreastCancerClassifier'.
2026/07/30 10:53:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BreastCancerClassifier, version 1


🏃 View run XGBoost_GridSearch_Registry_F1 at: http://127.0.0.1:5000/#/experiments/4/runs/4f52618e8ce74a7db029b080ea0a2868
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


Created version '1' of model 'BreastCancerClassifier'.


In [32]:
cv_results = pd.DataFrame(grid.cv_results_)
cv_results.sort_values("mean_test_score", ascending=False)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__learning_rate,param_model__max_depth,param_model__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
7,0.539892,0.033322,0.014397,0.000651,0.05,5,200,"{'model__learning_rate': 0.05, 'model__max_dep...",0.970588,0.985075,0.911765,1.000000,0.898551,0.953196,0.040527,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
11,0.311052,0.024187,0.012193,0.000474,0.10,5,200,"{'model__learning_rate': 0.1, 'model__max_dept...",0.955224,0.985075,0.911765,1.000000,0.898551,0.950123,0.039665,2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
10,0.274002,0.035614,0.013485,0.000835,0.10,5,100,"{'model__learning_rate': 0.1, 'model__max_dept...",0.955224,1.000000,0.911765,0.985075,0.898551,0.950123,0.039665,2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
8,0.205796,0.019517,0.015490,0.002341,0.10,3,100,"{'model__learning_rate': 0.1, 'model__max_dept...",0.939394,1.000000,0.911765,0.985075,0.885714,0.944390,0.043081,4,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
5,0.432296,0.066221,0.019923,0.002615,0.05,3,200,"{'model__learning_rate': 0.05, 'model__max_dep...",0.955224,0.985075,0.911765,0.969697,0.898551,0.944062,0.033401,5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
9,0.287942,0.029651,0.016777,0.001659,0.10,3,200,"{'model__learning_rate': 0.1, 'model__max_dept...",0.955224,0.985075,0.911765,0.969697,0.898551,0.944062,0.033401,5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
4,0.269595,0.015223,0.015513,0.001434,0.05,3,100,"{'model__learning_rate': 0.05, 'model__max_dep...",0.939394,1.000000,0.911765,0.985075,0.861111,0.939469,0.050302,7,0.992593,0.996310,0.996310,0.996310,0.996310,0.995566,0.001487
6,0.450744,0.075702,0.017649,0.001694,0.05,5,100,"{'model__learning_rate': 0.05, 'model__max_dep...",0.937500,1.000000,0.911765,0.985075,0.861111,0.939090,0.050308,8,1.000000,1.000000,1.000000,1.000000,0.996310,0.999262,0.001476
1,0.652094,0.103295,0.017713,0.001132,0.01,3,200,"{'model__learning_rate': 0.01, 'model__max_dep...",0.937500,0.985075,0.909091,0.955224,0.873239,0.932026,0.038377,9,0.985075,0.981413,0.992593,0.988848,0.988930,0.987371,0.003812
3,1.034159,0.088377,0.017110,0.001506,0.01,5,200,"{'model__learning_rate': 0.01, 'model__max_dep...",0.937500,0.970588,0.909091,0.970588,0.861111,0.929776,0.041313,10,0.988848,0.988848,0.996310,0.988848,0.992647,0.991100,0.002992


In [29]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
model_name = "BreastCancerClassifier"
model_version = 2
model_uri = f"models:/{model_name}/{model_version}"
mlflow_model = mlflow.sklearn.load_model(model_uri)
